<h1> Parsing + Chunking </h1>

In [ ]:
from unstructured.partition.text import partition_text #https://docs.unstructured.io/open-source/core-functionality/partitioning #txt files
from unstructured.partition.pdf import partition_pdf #pdf files
from unstructured.partition.docx import partition_docx #word files

from unstructured.chunking.title import chunk_by_title #https://docs.unstructured.io/open-source/core-functionality/chunking
from unstructured.chunking.basic import chunk_elements

from pathlib import Path
from langchain_core.documents import Document

In [2]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documentos = []
chunk_id = 1

for docs in path.iterdir():

    file_dtype = docs.suffix.lower()

    if file_dtype == ".pdf":

        parsing = partition_pdf (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)

    elif file_dtype == ".txt":

        parsing = partition_text (str(docs), languages = ["por"])
        chunks = chunk_elements (parsing, max_characters = 750)

    elif file_dtype == ".docx":        
        
        parsing = partition_docx (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)


    for chunk in chunks:

        documentos.append (
            Document (
                page_content = chunk.text,
                metadata = {
                    "title": docs.name,
                    "chunk_id": chunk_id
                }
            )
        )

        chunk_id += 1


#documentos


<hr>

<h1> Retrieval</h1>

<h3> Sparse Retrieval </h3>

In [15]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 15) #@K aqui definido

<hr>

<h3> Dense Retrieval </h3>

<h5> Embeddings & Indexing </h5>

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores #Não o melhor para produção


emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (documentos, emb_hf)

In [17]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 15
    }
)

<h3> Reciprocal Rank Fusion </h3>

$$
RRF (d) = \sum _{i = 1}^{n} \frac {1} {k + pos_i(d)} 
$$

In [48]:
from eval.HitRate import Reciprocal_Rank_Fusion

### EXEMPLO

query = "Em que consiste o relatório da rede EU Kids Online ?"

s_retrieval = sparse_retrieval.invoke (query) #Procura Lexical
d_retrieval = dense_retrieval.invoke (query) #Procura Vetorial

#display (s_retrieval)
#display (d_retrieval)

teste = Reciprocal_Rank_Fusion ([s_retrieval, d_retrieval])

#for ordem, (doc_name, chunk_id, chunk_text, score) in enumerate (teste, start = 1):
#    print (ordem)
#    print (doc_name)
#    print (chunk_id)
#    print (chunk_text)
#    print (score)

#display (len(teste))
#display (teste)


<h2> Eval Retrieval </h2>

In [46]:
#Dataset
import json

with open (r"C:\Users\Admin\Desktop\ip\How To RAG\eval\dataset.json", "r", encoding = "utf-8") as f:
    dataset = json.load(f)

from eval.HitRate import hitrate_k_sparse_retrieval, hitrate_k_dense_retrieval, hitrate_k_hybrid_retrieval
from eval.MeanReciprocalRank import mrr_sparse_retrieval, mrr_dense_retrieval, mrr_hybrid_retrieval


<h3> HitRate@5 </h3>

In [10]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 5)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 5
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

{'HitRate@K Chunk Sparse Retrieval': {0.49609375},
 'HitRate@K Docs Sparse Retrieval': {0.98828125}}

{'HitRate@K Chunk Dense Retrieval': {0.375},
 'HitRate@K Docs Dense Retrieval': {0.890625}}

{'HitRate@K Chunk Hybrid Retrieval': {0.671875},
 'HitRate@K Docs Hybrid Retrieval': {1.0}}

<h3> HitRate@10 </h3>

In [11]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 10)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

{'HitRate@K Chunk Sparse Retrieval': {0.6015625},
 'HitRate@K Docs Sparse Retrieval': {0.9921875}}

{'HitRate@K Chunk Dense Retrieval': {0.484375},
 'HitRate@K Docs Dense Retrieval': {0.91796875}}

{'HitRate@K Chunk Hybrid Retrieval': {0.75},
 'HitRate@K Docs Hybrid Retrieval': {1.0}}

<h3> HitRate@15 </h3>

In [12]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 15)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 15
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

{'HitRate@K Chunk Sparse Retrieval': {0.6640625},
 'HitRate@K Docs Sparse Retrieval': {0.99609375}}

{'HitRate@K Chunk Dense Retrieval': {0.5625},
 'HitRate@K Docs Dense Retrieval': {0.9296875}}

{'HitRate@K Chunk Hybrid Retrieval': {0.81640625},
 'HitRate@K Docs Hybrid Retrieval': {1.0}}

<h3> MRR </h3>

In [9]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 10)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

x = mrr_sparse_retrieval (sparse_retrieval_1, dataset)
y = mrr_dense_retrieval (dense_retrieval_1, dataset)
z = mrr_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

{'Mean Reciprocal Rank Chunk Sparse Retrieval': {0.38395492311507934},
 'Mean Reciprocal Rank Docs Sparse Retrieval': {0.9459635416666666}}

{'Mean Reciprocal Rank Chunks Dense Retrieval': {0.2799556671626985},
 'Mean Reciprocal Rank Docs Dense Retrieval': {0.8357406374007936}}

{'Mean Reciprocal Rank Chunks Hybrid Retrieval': {0.3886640247135422},
 'Mean Reciprocal Rank Docs Hybrid Retrieval': {0.9483506944444444}}

<hr>

<h6> código solto </h6>

In [ ]:
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores #Não o melhor para produção
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface

In [ ]:
emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (documentos, emb_hf)

In [ ]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

In [ ]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 10)

In [ ]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 20
    }
)

#x = dense_retrieval.invoke ("O que é a corrente dinâmica estipulada (Idyn)?")

In [ ]:
### EXEMPLO

query = "Em que consiste o relatório da rede EU Kids Online ?"

s_retrieval = sparse_retrieval.invoke (query)
d_retrieval = dense_retrieval.invoke (query)

teste = Reciprocal_Rank_Fusion ([s_retrieval, d_retrieval])

for ordem, (doc_name, chunk_id, chunk_text, score) in enumerate (teste, start = 1):
    print (ordem)
    print (doc_name)
    print (chunk_id)
    print (chunk_text)
    print (score)

#display (teste)


<h1> Reranking </h1>

In [20]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder_model = AutoModelForSequenceClassification.from_pretrained (path)
cross_encoder_model_tokenization = AutoTokenizer.from_pretrained (path)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2022.91it/s]


<h4> Cross Encoder </h4>

In [ ]:

query_reranker = [query] * len (teste)
chunks = [t[2] for t in teste]
docs_names = [t[0] for t in teste]
chunks_ids = [t[1] for t in teste]

pares = list(zip(query_reranker, chunks)) # [query, chunks]
#print (pares)

inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True)
#print (inputs)

cross_encoder_model.eval()
with torch.no_grad():
    logits = cross_encoder_model (**inputs).logits
    #print (logits)

#Organiza os logits com os chunks correspondentes #Muita Atenção! Para calcular HitRate e MRR com as funções criadas é necessário estar no mesmo formato do output do RRF.
rerank = sorted(zip(docs_names, chunks_ids, chunks, logits.tolist()), key = lambda x: x[3], reverse = True) #x[3] define que arg é usado pelo reverse

#display (rerank)


'Resumo:\n\nEste relatório da rede EU Kids Online apresenta usos e experiências de crianças e jovens com IA generativa, em Portugal, e como esta se integra atualmente nas suas vidas digitais. Inclui resultados do inquérito nacional a 2.111 crianças e jovens de 9 a 17 anos, a que se juntam testemunhos de 15 adolescentes (13-17 anos) que fazem uso de ferramentas de IA generativa.\n\nO estudo em que assenta integra-se nos projetos estratégicos do Instituto de Comunicação da Nova (ICNOVA), UIDB/05021/2025 e do Centro Interdisciplinar de Ciências Sociais da Universidade Nova de Lisboa (CICS.NOVA), UID/PRR/04647/2025.'

<h4> Ranking </h4>

In [ ]:
final = [(docs, chunks, chunks_content, scores) for docs, chunks, chunks_content, scores in rerank [:5]]

#display (final[1])

<h3> Eval RAG + Reranking </h3>

In [ ]:
from eval.MeanReciprocalRank import mrr_ranker_sparse_system, mrr_ranker_dense_system, mrr_ranker_hybrid_system

f = mrr_ranker_sparse_system (sparse_retrieval, dataset)
h = mrr_ranker_dense_system (dense_retrieval, dataset)
i = mrr_ranker_hybrid_system (sparse_retrieval, dense_retrieval, dataset)

display (f)
display (h)
display (i)


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2781.05it/s]


{'Mean Reciprocal Rank Chunks Rerank': {0.4693359374999999},
 'Mean Reciprocal Rank Docs Rerank': {0.9698567708333334}}

{'Mean Reciprocal Rank Chunks Rerank': {0.3710286458333334},
 'Mean Reciprocal Rank Docs Rerank': {0.8995442708333333}}

{'Mean Reciprocal Rank Chunks Rerank': {0.5242838541666667},
 'Mean Reciprocal Rank Docs Rerank': {0.9744140625000001}}

<h1> Eval Pré LLM </h1>

In [58]:

#Cross Encoder Model para Rerank
path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder_model = AutoModelForSequenceClassification.from_pretrained (path)
cross_encoder_model_tokenization = AutoTokenizer.from_pretrained (path)

#Definição de Sparse Retrieval
sparse_retrieval_pre_llm = BM25Retriever.from_documents (documentos, k = 15) #@K aqui definido

#Definição de Dense Retrieval
dense_retrieval_pre_llm = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 15
    }
)

dados = []

for data in dataset:

    q = data["query"]

    #print (q)
    sp_retr = sparse_retrieval_pre_llm.invoke (q)
    den_retr = dense_retrieval_pre_llm.invoke (q)
    
    rrf = Reciprocal_Rank_Fusion ([sp_retr, den_retr])

    query_reranker = [q] * len (rrf)
    chunks = [t[2] for t in rrf]
    docs_names = [t[0] for t in rrf]
    chunks_ids = [t[1] for t in rrf]

    pares = list(zip(query_reranker, chunks)) # [query, chunks]
    #print (pares)

    inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True)
    #print (inputs)

    cross_encoder_model.eval()
    with torch.no_grad():
        logits = cross_encoder_model (**inputs).logits
        #print (logits)

    #Organiza os logits com os chunks correspondentes #Muita Atenção! Para calcular HitRate e MRR com as funções criadas é necessário estar no mesmo formato do output do RRF.
    rerank = sorted(zip(chunks, logits.tolist()), key = lambda x: x[1], reverse = True) #x[3] define que arg é usado pelo reverse

    rank = [(chunks, scores) for chunks, scores in rerank [:8]]

    dados.append (q)
    dados.append (rank)


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1671.04it/s]


In [61]:
dados

['O que são máquinas elétricas no dia-a-dia?',
 [('ABC DAS MÁQUINAS ELÉCTRICAS\n\nMário Ferreira Alves (malves@dee.isep.ipp.pt)\n\nDepartamento de Engenharia Electrotécnica\n\nMarço de 2003\n\nPrefácio\n\nActualmente, podemos considerar as máquinas eléctricas (motores, geradores e transformadores) como parte integrante do nosso dia-a-dia. Os motores eléctricos, que podem utilizar-se tanto em aplicações de força motriz como em aplicações de tracção eléctrica, vulgarizaram-se de tal forma que podemos encontrá-los em aplicações tão diversas como uma máquina industrial de corte, um ascensor ou um aspirador.',
   [2.6109933853149414]),
  ('“É como se ele estivesse a falar diretamente só sobre nós, ao contrário de como [se] fôssemos falar com uma pessoa no dia a dia: ela também falaria da vida delas, iria falar sobre coisas mais pessoais sobre nós e sobre ela.” (Ana, 14)\n\n"Uma ferramenta... é um robô, é mais direto e mais seco assim. E uma pessoa não. Ela vai falar, vai reagir, tem sentime

<h4> Context Recall </h4>

In [68]:
results = [1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


print (sum(results) / len(results))

0.8771929824561403


<h4> Context Precision </h4>

In [75]:
results = [1/8, 2/8, 2/8, 0/8, 2/8, 0/8, 2/8, 4/8, 8/8, 3/8, 2/8, 2/8, 2/8, 2/8, 1/8, 4/8, 2/8]

print (sum(results) / len(results)) 

0.2867647058823529


<h3> LLM </h3>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

path = r"C:\Users\Admin\Desktop\models\Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer2 = AutoTokenizer.from_pretrained (path)

device


In [ ]:
prompt = "Em que consiste o relatório da rede EU Kids Online ?"


chat_template = f"""
<|im_start|>
És um assistente especializado em responder exclusivamente com base no contexto fornecido.

Regras:
1. Utiliza apenas a informação do Contexto.
2. Não inventes informação.
3. Não uses conhecimento externo.
4. Se a resposta não existir no contexto, responde exatamente:
   "Informação não encontrada na base de dados."
5. Responde de forma breve e direta.
6. Formata a resposta de forma clara.

Contexto:
{feed_llm}
<|im_end|>

<|im_start>
{prompt}
<|im_end|>

<|im_start|>
"""

model_inputs = tokenizer2 (chat_template, return_tensors = "pt").to(model.device)

generated_ids = model.generate(**model_inputs, max_new_tokens = 512)

input_length = model_inputs["input_ids"].shape[1]
response_tokens = generated_ids[0][input_length:]

print(tokenizer2.decode(response_tokens, skip_special_tokens = True))

<hr>

In [ ]:
mrr_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    sparse_retr = sparse_retrieval.invoke(query)

    rr = 0

    for rank, doc in enumerate(dense_retr, start=1):
        if doc.metadata["title"] == gold_doc:
            rr = 1 / rank
            break

    mrr_scores.append(rr)

mrr = sum(mrr_scores) / len(mrr_scores)

print("MRR Sparse:", mrr)

In [ ]:
mrr_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    dense_retr = dense_retrieval.invoke(query)

    rr = 0

    for rank, doc in enumerate(dense_retr, start=1):
        if doc.metadata["title"] == gold_doc:
            rr = 1 / rank
            break

    mrr_scores.append(rr)

mrr = sum(mrr_scores) / len(mrr_scores)

print("MRR Dense:", mrr)

In [ ]:
rrf_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    rankings = [
        dense_retrieval.invoke(query),
        sparse_retrieval.invoke(query)
    ]

    rrf_ranked = Reciprocal_Rank_Fusion(rankings)

    rr = 0

    for rank, (doc_content, score, doc_id) in enumerate(rrf_ranked, start=1):
        if doc_id == gold_doc:
            rr = 1 / rank
            break

    rrf_scores.append(rr)

mrr_rrf = sum(rrf_scores) / len(rrf_scores)

print("MRR RRF:", mrr_rrf)